<h1 style="display:flex; align-items:center; line-height:1; margin:0;">
  <img src="rwcircle.png" style="height:36px; width:36px; border-radius:50%; flex-shrink:0;">
  <span style="padding-top:2px; margin-left:10px;"><b>2. Data Challenge Submission</b>: an end-to-end primer</span>
</h1>

***
**Author**: Nestor Espinoza (nespinoza@stsci.edu) | **Date of last change**: June 30, 2026

## Summary
In this notebook, we perform a simple study of a subset of the Rocky Worlds DDT Data Challenge datasets, which we take and study. We package results into a `.zip` file which we can then submit to the Data Challenge at the end of the Notebook.

## Pre-study imports and definitions
First, let's load useful libraries for this notebook:

In [ ]:
import glob

import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import median_filter

from scipy.optimize import least_squares
import ipywidgets as widgets
from IPython.display import display, clear_output

import batman
import emcee
import corner
from jwst import datamodels

**Important:** if you did not download the 40 GB of data that notebook 1 of this tutorial downloads (`1-download-some-data.ipynb`), turn the following variable to `False` so you use the self-contained time-series in this Notebook instead:

In [ ]:
have_uncal_data = True

# 1. From detector ramps to light curves

<div class="alert alert-danger">
<b>Beware:</b> The data analysis performed below ommits several important data analysis steps that give rise to much better precision lightcurves. Most of those are implemented in packages such as the <a href="https://jwst-pipeline.readthedocs.io/en/stable/" target="_blank">JWST calibration pipeline</a>. We do not recommend following the data reduction procedures described below for real science, or even for "real/serious" <a href="https://www.kaggle.com/competitions/rocky-worlds-data-challenge" target="_blank">Rocky Worlds DDT Data Challenge</a> submissions. It is in this notebook, however, for getting started, or for having quicklooks at the data.
</div>

## 1.1 Studying GJ 3929 b's Data Challenge data

Let's do a simple last-minus-first study of GJ 3929 b's eclipse 3. First, load the data --- and do a simple last-minus-first to obtain an estimate of the rates per integration:

In [ ]:
if have_uncal_data:
    
    files = glob.glob('rocky_worlds/gj3929b/eclipse_03/*.fits')
    files.sort()
    
    def get_times_and_rateints(files):
        
        counter = 0
        for f in files:
        
            dataset = datamodels.RampModel(f)
            ramp = dataset.data
        
            if counter == 0:
                
                times = dataset.int_times['int_mid_BJD_TDB'] + 2400000.5
                rateints = ramp[:, -2, :, :] - ramp[:, 1, :, :]
        
            else:
        
                times = np.append(times, dataset.int_times['int_mid_BJD_TDB'] + 2400000.5)
                rateints = np.vstack(( rateints, ramp[:, -2, :, :] - ramp[:, 1, :, :] ))
        
            counter +=1
    
        return times, rateints
    
    times_gj, rateints_gj = get_times_and_rateints(files)

Let's see how the median PSF looks like:

In [ ]:
if have_uncal_data:
    
    median_gj = np.nanmedian( rateints_gj, axis = (0) )
    
    im = plt.imshow(median_gj)
    im.set_clim(650,800)
    plt.show()

Let's extract some ultra-crude photometry:

In [ ]:
if have_uncal_data:

    photometry_gj = np.nansum(rateints_gj[:, 50:75, 60:75], axis = (1,2))
    relative_photometry_gj = photometry_gj / np.nanmedian(photometry_gj)

Let's plot it:

In [ ]:
if have_uncal_data:

    plt.figure(figsize = (10,4))
    
    relative_time_gj = ( times_gj - times_gj[0] ) * 24.
    
    plt.plot( relative_time_gj, relative_photometry_gj)
    
    
    plt.title('Relative photometry for GJ 3929 b --- Eclipse 3', fontsize = 16)
    plt.xlabel('Time from exposure start (hours)', fontsize = 16)
    plt.ylabel('Relative flux', fontsize = 16)
    
    plt.xticks(fontsize = 14)
    plt.yticks(fontsize = 14)
    
    plt.ylim( 0.996,1.004 )
    plt.xlim( np.min(relative_time_gj), np.max(relative_time_gj) )
    plt.show()

Let's try to correct some obvious outliers via a simple median-filter-based outlier rejection algorithm:

In [ ]:
def correct_outliers(data, window = 51, nsigma = 5):
    
    # Estimate the median filter of the data:
    mf = median_filter(data, window)

    # With it, estimate scatter of the data via the residuals using the Median Absolute Deviation (MAD):
    residuals = data - mf
    mad = np.median( np.abs(residuals - np.nanmedian(residuals)) )
    sigma = 1.4 * mad

    # Identify outliers; replace them by the median filter:
    idx = np.where( np.abs( residuals ) > nsigma * sigma )[0]
    corrected_data = np.copy(data)
    corrected_data[idx] = mf[idx]
    
    return corrected_data

Let's see how we do:

In [ ]:
if have_uncal_data:
    
    plt.figure(figsize = (10,4))
    
    plt.title('Relative photometry for GJ 3929 b --- Eclipse 3', fontsize = 16)
    corrected_relative_photometry_gj = correct_outliers(relative_photometry_gj)
    
    plt.plot( relative_time_gj, relative_photometry_gj)
    plt.plot( relative_time_gj, corrected_relative_photometry_gj)
    
    plt.xlabel('Time from exposure start (hours)', fontsize = 16)
    plt.ylabel('Relative flux', fontsize = 16)
    
    plt.xticks(fontsize = 14)
    plt.yticks(fontsize = 14)
    
    plt.ylim( 0.996,1.004 )
    plt.xlim( np.min(relative_time_gj), np.max(relative_time_gj) )
    plt.show()

In [ ]:
if not have_uncal_data:

    times_gj, corrected_relative_photometry_gj = np.loadtxt('ts_gj.txt', unpack = True, usecols = (0, 1))

## 1.2 Studying LHS 1140 b's (simulated) Data Challenge data

And now let's repeat the exercise for LHS 1140 b's dataset. First, get eclipses:

In [ ]:
if have_uncal_data:

    files = glob.glob('rocky_worlds/lhs1140b/eclipse_01/*.fits')
    files.sort()

Get ramps:

In [ ]:
if have_uncal_data:
    
    times_lhs, rateints_lhs = get_times_and_rateints(files)

See median frame:

In [ ]:
if have_uncal_data:

    median_lhs = np.nanmedian( rateints_lhs, axis = (0) )
    
    im = plt.imshow(median_lhs)
    im.set_clim(1800,2100)
    plt.show()

In [ ]:
if have_uncal_data:

    im = plt.imshow(median_lhs)
    im.set_clim(1800,2100)
    plt.xlim(100,150)
    plt.ylim(100,150)
    plt.show()

Photometry:

In [ ]:
if have_uncal_data:

    photometry_lhs = np.nansum(rateints_lhs[:, 115:140, 115:140], axis = (1,2))
    relative_photometry_lhs = photometry_lhs / np.nanmedian(photometry_lhs)

Correct in advance:

In [ ]:
if have_uncal_data:
    
    corrected_relative_photometry_lhs = correct_outliers(relative_photometry_lhs)

In [ ]:
if have_uncal_data:

    plt.figure(figsize = (10,4))
    
    relative_time_lhs = ( times_lhs - times_lhs[0] ) * 24.
    
    plt.plot(relative_time_lhs, relative_photometry_lhs)
    plt.plot(relative_time_lhs, corrected_relative_photometry_lhs)
    
    
    plt.title('Relative photometry for (simulated) LHS 1140 b --- Eclipse 1', fontsize = 16)
    
    plt.xlabel('Time from exposure start (hours)', fontsize = 16)
    plt.ylabel('Relative flux', fontsize = 16)
    
    plt.xticks(fontsize = 14)
    plt.yticks(fontsize = 14)
    
    plt.ylim( 0.996,1.004 )
    plt.xlim( np.min(relative_time_gj), np.max(relative_time_gj) )
    plt.show()

In [ ]:
if not have_uncal_data:

    times_lhs, corrected_relative_photometry_lhs = np.loadtxt('ts_lhs.txt', unpack = True, usecols = (0, 1))

# 2. Fitting light curves

## 2.1 Fitting GJ 3929 b's eclipse 3
Let's next do a very crude fitting of these lightcurves. Let's start with defining a big function that would allow us to do all this interactively:

In [ ]:
def exponential_function(t, exp_amplitude, exp_timescale, slope):
    return (
        1.0 - exp_amplitude * np.exp(-(t - t[0]) / exp_timescale)
    ) * (
        1.0 - slope * (t - t[0])
    )


def bin_by_points(t, y, bin_size=1):
    t = np.asarray(t)
    y = np.asarray(y)

    if bin_size <= 1:
        return t.copy(), y.copy(), np.ones_like(y, dtype=int)

    nbins = len(t) // bin_size

    if nbins < 1:
        return t.copy(), y.copy(), np.ones_like(y, dtype=int)

    t_trim = t[:nbins * bin_size]
    y_trim = y[:nbins * bin_size]

    t_binned = t_trim.reshape(nbins, bin_size).mean(axis=1)
    y_binned = y_trim.reshape(nbins, bin_size).mean(axis=1)
    n_per_bin = np.ones(nbins, dtype=int) * bin_size

    remainder = len(t) - nbins * bin_size

    if remainder > 0:
        t_binned = np.append(t_binned, t[nbins * bin_size:].mean())
        y_binned = np.append(y_binned, y[nbins * bin_size:].mean())
        n_per_bin = np.append(n_per_bin, remainder)

    return t_binned, y_binned, n_per_bin


class InteractiveEclipseFitter:
    def __init__(
        self,
        time,
        flux,
        t0=0.0,
        period=1.0,
        a_rs=10.0,
        inc=89.0,
        rp=0.1,
        initial=None,
        max_bin_size=100,
    ):
        self.t_raw = np.asarray(time)
        self.f_raw = np.asarray(flux)

        idx = np.argsort(self.t_raw)
        self.t_raw = self.t_raw[idx]
        self.f_raw = self.f_raw[idx]

        self.t0 = t0
        self.period = period
        self.a_rs = a_rs
        self.inc = inc
        self.rp = rp

        tmid = np.median(self.t_raw)
        self.ncycle = int(
            np.round((tmid - self.t0 - 0.5 * self.period) / self.period)
        )

        self.max_bin_size = max_bin_size

        self.samples = None
        self.sample_logprob = None
        self.posterior_best_fit = None
        self.posterior = None

        self.parameter_keys = None
        self.rw_samples = None
        self.rw_photometry = None

        if initial is None:
            initial = dict(
                secosomega=0.0,
                sesinomega=0.0,
                fp=500e-6,
                C=1.0,
                exp_amplitude=0.001,
                exp_timescale=0.2,
                slope=0.0,
                jitter=np.nanstd(self.f_raw),
                bin_size=1,
            )

        self.sliders = {
            "secosomega": widgets.FloatSlider(
                value=initial["secosomega"],
                min=-1.0,
                max=1.0,
                step=0.001,
                description="√e cosω",
                continuous_update=False,
                readout_format=".4f",
            ),
            "sesinomega": widgets.FloatSlider(
                value=initial["sesinomega"],
                min=-1.0,
                max=1.0,
                step=0.001,
                description="√e sinω",
                continuous_update=False,
                readout_format=".4f",
            ),
            "fp": widgets.FloatLogSlider(
                value=initial["fp"],
                base=10,
                min=-7,
                max=-2,
                step=0.01,
                description="Depth",
                continuous_update=False,
                readout_format=".2e",
            ),
            "C": widgets.FloatSlider(
                value=initial["C"],
                min=0.98,
                max=1.02,
                step=1e-5,
                description="C",
                continuous_update=False,
                readout_format=".6f",
            ),
            "exp_amplitude": widgets.FloatSlider(
                value=initial["exp_amplitude"],
                min=-0.02,
                max=0.02,
                step=1e-5,
                description="Exp amp",
                continuous_update=False,
                readout_format=".5f",
            ),
            "exp_timescale": widgets.FloatLogSlider(
                value=initial["exp_timescale"],
                base=10,
                min=-3,
                max=1,
                step=0.01,
                description="Exp tau",
                continuous_update=False,
                readout_format=".4f",
            ),
            "slope": widgets.FloatSlider(
                value=initial["slope"],
                min=-0.02,
                max=0.02,
                step=1e-6,
                description="Slope",
                continuous_update=False,
                readout_format=".6f",
            ),
            "jitter": widgets.FloatLogSlider(
                value=initial["jitter"],
                base=10,
                min=-7,
                max=-1,
                step=0.01,
                description="Jitter",
                continuous_update=False,
                readout_format=".2e",
            ),
            "bin_size": widgets.IntSlider(
                value=initial["bin_size"],
                min=1,
                max=max_bin_size,
                step=1,
                description="Bin size",
                continuous_update=False,
            ),
        }

        self.fit_button = widgets.Button(description="Fit from sliders")
        self.mcmc_button = widgets.Button(description="Run emcee")
        self.output = widgets.Output()

        self.reconnect_slider_updates()

        self.fit_button.on_click(self.run_fit)
        self.mcmc_button.on_click(self.run_emcee)

        controls = widgets.VBox([
            widgets.HBox([self.sliders["secosomega"], self.sliders["sesinomega"]]),
            widgets.HBox([self.sliders["fp"], self.sliders["C"]]),
            widgets.HBox([self.sliders["exp_amplitude"], self.sliders["exp_timescale"]]),
            widgets.HBox([self.sliders["slope"], self.sliders["jitter"]]),
            widgets.HBox([self.sliders["bin_size"], self.fit_button, self.mcmc_button]),
        ])

        display(controls, self.output)
        self.update()

    def disconnect_slider_updates(self):
        for s in self.sliders.values():
            try:
                s.unobserve(self.update, names="value")
            except ValueError:
                pass

    def reconnect_slider_updates(self):
        for s in self.sliders.values():
            try:
                s.observe(self.update, names="value")
            except ValueError:
                pass

    def set_sliders_safely(self, p):
        self.disconnect_slider_updates()

        for k, v in p.items():
            if k in self.sliders:
                self.sliders[k].value = v

        self.reconnect_slider_updates()

    def get_binned_data(self):
        return bin_by_points(
            self.t_raw,
            self.f_raw,
            bin_size=self.sliders["bin_size"].value,
        )

    def get_values(self):
        return {
            k: s.value
            for k, s in self.sliders.items()
            if k != "bin_size"
        }

    def eclipse_time(self, secosomega, sesinomega):
        omega = np.arctan2(sesinomega, secosomega)
        ecc = secosomega**2 + sesinomega**2

        if ecc >= 1.0:
            ecc = 0.999999

        ecosomega = ecc * np.cos(omega)

        t_secondary = (
            self.t0
            + (self.ncycle + 0.5) * self.period
            + self.period * (2.0 / np.pi) * ecosomega
        )

        return t_secondary, ecc, omega

    def eclipse_model(self, t, secosomega, sesinomega, fp):
        t_secondary, ecc, omega = self.eclipse_time(secosomega, sesinomega)

        params = batman.TransitParams()
        params.t0 = self.t0
        params.per = self.period
        params.rp = self.rp
        params.a = self.a_rs
        params.inc = self.inc
        params.ecc = ecc
        params.w = np.degrees(omega)
        params.u = []
        params.limb_dark = "uniform"

        params.fp = fp
        params.t_secondary = t_secondary

        m = batman.TransitModel(params, t, transittype="secondary")
        eclipse = m.light_curve(params)

        return eclipse, t_secondary, ecc, omega

    def model_components(self, t, p):
        astro_model, tsec, ecc, omega = self.eclipse_model(
            t,
            p["secosomega"],
            p["sesinomega"],
            p["fp"],
        )

        baseline = exponential_function(
            t,
            p["exp_amplitude"],
            p["exp_timescale"],
            p["slope"],
        )

        noise_model = p["C"] * baseline
        full_model = astro_model * noise_model

        return astro_model, noise_model, full_model, tsec, ecc, omega

    def full_model(self, t, p):
        astro_model, noise_model, full_model, tsec, ecc, omega = self.model_components(t, p)
        return full_model, tsec, ecc, omega

    def chi2red(self, model, y, yerr, npars=8):
        chi2 = np.sum(((y - model) / yerr) ** 2)
        dof = len(y) - npars

        if dof <= 0:
            return np.nan

        return chi2 / dof

    def parameter_text(self, p, tsec, ecc, omega, chi2r, title="Current model"):
        return (
            f"{title}\n\n"
            f"secosw = {p['secosomega']:.5f}\n"
            f"sesinw = {p['sesinomega']:.5f}\n"
            f"e = {ecc:.5f}\n"
            f"ω = {np.degrees(omega):.2f} deg\n"
            f"ncycle = {self.ncycle}\n"
            f"t_sec = {tsec:.8f}\n"
            f"depth_ecl = {1e6 * p['fp']:.2f} ppm\n"
            f"C = {p['C']:.7f}\n"
            f"exp_amp = {p['exp_amplitude']:.5e}\n"
            f"tau = {p['exp_timescale']:.5f}\n"
            f"slope = {p['slope']:.5e}\n"
            f"jitter = {p['jitter']:.3e}\n"
            f"bin size = {self.sliders['bin_size'].value}\n"
            f"χ²_red = {chi2r:.3f}"
        )

    def plot_solution(self, p, model_label="Current model", title="Current model"):
        t, f, n_per_bin = self.get_binned_data()
        yerr = p["jitter"] / np.sqrt(n_per_bin)

        model, tsec, ecc, omega = self.full_model(t, p)
        residuals = f - model
        chi2r = self.chi2red(model, f, yerr)

        fig = plt.figure(figsize=(13, 6))
        gs = fig.add_gridspec(
            2,
            2,
            width_ratios=[4.5, 1.6],
            height_ratios=[3, 1],
            wspace=0.28,
            hspace=0.08,
        )

        ax_lc = fig.add_subplot(gs[0, 0])
        ax_res = fig.add_subplot(gs[1, 0], sharex=ax_lc)
        ax_txt = fig.add_subplot(gs[:, 1])

        ax_lc.errorbar(
            t,
            f,
            yerr=yerr,
            fmt=".",
            alpha=0.8,
            label="Binned data",
        )

        ax_lc.plot(
            t,
            model,
            lw=2,
            label=model_label,
        )

        ax_lc.set_ylabel("Relative flux")
        ax_lc.legend(loc="best")

        ax_res.axhline(0.0, lw=1)
        ax_res.errorbar(
            t,
            residuals,
            yerr=yerr,
            fmt=".",
            alpha=0.8,
        )

        ax_res.set_xlabel("Time [BJD]")
        ax_res.set_ylabel("Residuals")

        ax_txt.axis("off")
        ax_txt.text(
            0.0,
            1.0,
            self.parameter_text(p, tsec, ecc, omega, chi2r, title=title),
            va="top",
            ha="left",
            fontsize=10,
            family="monospace",
        )

        plt.setp(ax_lc.get_xticklabels(), visible=False)

        return fig

    def update(self, *_):
        p = self.get_values()

        with self.output:
            clear_output(wait=False)

            fig = self.plot_solution(
                p,
                model_label="Current model",
                title="Current model",
            )

            plt.show()
            plt.close(fig)

    def pack(self):
        p = self.get_values()

        return np.array([
            p["secosomega"],
            p["sesinomega"],
            p["fp"],
            p["C"],
            p["exp_amplitude"],
            p["exp_timescale"],
            p["slope"],
            p["jitter"],
        ])

    def unpack(self, x):
        return dict(
            secosomega=x[0],
            sesinomega=x[1],
            fp=x[2],
            C=x[3],
            exp_amplitude=x[4],
            exp_timescale=x[5],
            slope=x[6],
            jitter=x[7],
        )

    def log_prior(self, x):
        p = self.unpack(x)

        ecc = p["secosomega"]**2 + p["sesinomega"]**2

        if ecc >= 1.0:
            return -np.inf

        if not (-1.0 <= p["secosomega"] <= 1.0):
            return -np.inf
        if not (-1.0 <= p["sesinomega"] <= 1.0):
            return -np.inf
        if not (1e-7 <= p["fp"] <= 1e-2):
            return -np.inf
        if not (0.95 <= p["C"] <= 1.05):
            return -np.inf
        if not (-0.05 <= p["exp_amplitude"] <= 0.05):
            return -np.inf
        if not (1e-4 <= p["exp_timescale"] <= 10.0):
            return -np.inf
        if not (-0.1 <= p["slope"] <= 0.1):
            return -np.inf
        if not (1e-7 <= p["jitter"] <= 1e-1):
            return -np.inf

        return 0.0

    def log_likelihood(self, x):
        p = self.unpack(x)

        t, f, n_per_bin = self.get_binned_data()
        yerr = p["jitter"] / np.sqrt(n_per_bin)

        model, _, _, _ = self.full_model(t, p)
        r = (f - model) / yerr

        return -0.5 * np.sum(
            r**2 + np.log(2.0 * np.pi * yerr**2)
        )

    def log_probability(self, x):
        lp = self.log_prior(x)

        if not np.isfinite(lp):
            return -np.inf

        return lp + self.log_likelihood(x)

    def residual_vector_for_fit(self, xfit):
        x = xfit.copy()
        x[7] = 10.0 ** xfit[7]

        p = self.unpack(x)

        t, f, n_per_bin = self.get_binned_data()

        if p["secosomega"]**2 + p["sesinomega"]**2 >= 1.0:
            return np.ones_like(f) * 1e10

        yerr = p["jitter"] / np.sqrt(n_per_bin)
        model, _, _, _ = self.full_model(t, p)

        return (f - model) / yerr

    def run_fit(self, *_):
        x0 = self.pack()

        x0_fit = x0.copy()
        x0_fit[7] = np.log10(x0[7])

        lower = np.array([
            -1.0,
            -1.0,
            1e-7,
            0.95,
            -0.05,
            1e-4,
            -0.1,
            -7.0,
        ])

        upper = np.array([
            1.0,
            1.0,
            1e-2,
            1.05,
            0.05,
            10.0,
            0.1,
            -1.0,
        ])

        with self.output:
            clear_output(wait=False)
            print("Running least-squares fit...")

        result = least_squares(
            self.residual_vector_for_fit,
            x0_fit,
            bounds=(lower, upper),
            method="trf",
            max_nfev=5000,
        )

        x_best = result.x.copy()
        x_best[7] = 10.0 ** result.x[7]

        pfit = self.unpack(x_best)

        t, f, n_per_bin = self.get_binned_data()
        model, _, _, _ = self.full_model(t, pfit)
        residuals = f - model

        pfit["jitter"] = np.std(residuals * np.sqrt(n_per_bin))
        pfit["jitter"] = np.clip(pfit["jitter"], 1e-7, 1e-1)

        self.posterior_best_fit = pfit

        self.set_sliders_safely(pfit)
        self.make_rw_photometry_outputs(pfit)

        self.update()

    def run_emcee(self, *_):
        ndim = 8
        nwalkers = 100
        nsteps = 500
        nsamples_final = 10_000

        x_best = self.pack()

        scale = np.array([
            1e-3,
            1e-3,
            0.05 * max(x_best[2], 1e-6),
            1e-5,
            1e-4,
            0.05 * max(x_best[5], 1e-3),
            1e-5,
            0.05 * max(x_best[7], 1e-7),
        ])

        p0 = x_best + scale * np.random.randn(nwalkers, ndim)

        for i in range(nwalkers):
            tries = 0

            while not np.isfinite(self.log_prior(p0[i])):
                p0[i] = x_best + scale * np.random.randn(ndim)
                tries += 1

                if tries > 1000:
                    raise RuntimeError("Could not initialize walkers inside prior.")

        with self.output:
            clear_output(wait=False)
            print("Running emcee...")
            print(f"Initial log-probability: {self.log_probability(x_best):.3f}")

        sampler = emcee.EnsembleSampler(
            nwalkers,
            ndim,
            self.log_probability,
        )

        sampler.run_mcmc(p0, nsteps, progress=False)

        chain = sampler.get_chain(flat=True)
        logprob = sampler.get_log_prob(flat=True)

        self.samples = chain[-nsamples_final:]
        self.sample_logprob = logprob[-nsamples_final:]

        self.secosw_posterior = self.samples[:, 0]
        self.sesinw_posterior = self.samples[:, 1]
        self.depth_ecl_posterior = 1e6 * self.samples[:, 2]
        self.C_posterior = self.samples[:, 3]
        self.exp_amplitude_posterior = self.samples[:, 4]
        self.exp_timescale_posterior = self.samples[:, 5]
        self.slope_posterior = self.samples[:, 6]
        self.jitter_posterior = self.samples[:, 7]

        self.posterior = {
            "depth_ecl": self.depth_ecl_posterior,
            "secosw": self.secosw_posterior,
            "sesinw": self.sesinw_posterior,
            "C": self.C_posterior,
            "exp_amplitude": self.exp_amplitude_posterior,
            "exp_timescale": self.exp_timescale_posterior,
            "slope": self.slope_posterior,
            "jitter": self.jitter_posterior,
        }

        self.make_rw_posterior_outputs()

        imax = np.argmax(self.sample_logprob)
        x_map = self.samples[imax]
        p_map = self.unpack(x_map)

        self.posterior_best_fit = p_map

        self.set_sliders_safely(p_map)
        self.make_rw_photometry_outputs(p_map)

        self.show_mcmc_results()

    def show_mcmc_results(self):
        p = self.posterior_best_fit

        with self.output:
            clear_output(wait=False)

            fig = self.plot_solution(
                p,
                model_label="Posterior best-fit",
                title="Posterior MAP",
            )

            plt.show()
            plt.close(fig)

            corner_samples = np.column_stack([
                self.secosw_posterior,
                self.sesinw_posterior,
                self.depth_ecl_posterior,
            ])

            fig_corner = corner.corner(
                corner_samples,
                labels=[
                    r"$\sqrt{e}\cos\omega$",
                    r"$\sqrt{e}\sin\omega$",
                    r"$\mathrm{depth}_{\mathrm{ecl}}$ [ppm]",
                ],
                truths=[
                    p["secosomega"],
                    p["sesinomega"],
                    1e6 * p["fp"],
                ],
                show_titles=True,
                title_fmt=".3f",
            )

            plt.show()
            plt.close(fig_corner)

    def make_rw_posterior_outputs(self):
        if self.samples is None:
            raise RuntimeError("Run emcee before creating Rocky Worlds posterior outputs.")

        self.parameter_keys = [
            "depth_ecl",
            "secosw",
            "sesinw",
        ]

        # Shape expected by Rocky Worlds: (n_parameters, n_posterior_samples)
        self.rw_samples = np.vstack([
            1e6 * self.samples[:, 2],
            self.samples[:, 0],
            self.samples[:, 1],
        ])

        return self.rw_samples, self.parameter_keys

    def make_rw_photometry_outputs(self, p=None):
        if p is None:
            if self.posterior_best_fit is not None:
                p = self.posterior_best_fit
            else:
                p = self.get_values()

        astro_model, noise_model, full_model, _, _, _ = self.model_components(
            self.t_raw,
            p,
        )

        raw_flux_err = np.ones_like(self.f_raw) * p["jitter"]

        self.rw_photometry = {
            "time": self.t_raw.copy(),
            "raw_flux": self.f_raw.copy(),
            "raw_flux_err": raw_flux_err,
            "astro_model": astro_model,
            "noise_model": noise_model,
            "full_model": full_model,
        }

        return self.rw_photometry

    def get_rw_outputs(self):
        if self.rw_samples is None or self.parameter_keys is None:
            self.make_rw_posterior_outputs()

        if self.rw_photometry is None:
            self.make_rw_photometry_outputs()

        return {
            "samples": self.rw_samples,
            "parameter_keys": self.parameter_keys,
            "photometry": self.rw_photometry,
        }

    def make_rw_objects(self, rw):
        outputs = self.get_rw_outputs()

        posterior = rw.Posterior(
            samples=outputs["samples"],
            parameter_keys=outputs["parameter_keys"],
        )

        photometry = rw.Photometry(**outputs["photometry"])

        return posterior, photometry

Define some parameters for GJ 3929 b (taken from the NASA Exoplanet Archive):

In [ ]:
t0_gj = 2458956.3962
P_gj = 2.6162745
a_rs_gj = 17.56
inc_gj = 89.65
rp_gj = 0.03348

Get to the fit:

In [ ]:
fitter_gj = InteractiveEclipseFitter(
    times_gj,
    corrected_relative_photometry_gj,
    t0=t0_gj,
    period=P_gj,
    a_rs=a_rs_gj,
    inc=inc_gj,
    rp=rp_gj
)

Cool! Note we can access the samples for the important parameters over here:

In [ ]:
fitter_gj.posterior["depth_ecl"]
fitter_gj.posterior["secosw"]
fitter_gj.posterior["sesinw"]

Or, you can extract directly in Rocky Worlds DDT Data Challenge-like products:

In [ ]:
outputs_gj = fitter_gj.get_rw_outputs()

samples_gj = outputs_gj["samples"]
parameter_keys_gj = outputs_gj["parameter_keys"]
photometry_kwargs_gj = outputs_gj["photometry"]

In [ ]:
photometry_kwargs_gj

We will use those later.

## 2.2 Fitting LHS 1140 b's (simulated) eclipse 1

Let's now try to fit LHS 1140 b's (simulated) eclipse 1 with the very same function. First, some input parameters we take from the [NASA Exoplanet Archive](https://exoplanetarchive.ipac.caltech.edu/overview/LHS%201140):

In [ ]:
t0_lhs = 2458399.9300
P_lhs = 24.73723
a_rs_lhs = 95.3
inc_lhs = 89.86
rp_lhs = 0.07390

In [ ]:
fitter_lhs = InteractiveEclipseFitter(
    times_lhs,
    corrected_relative_photometry_lhs,
    t0=t0_lhs,
    period=P_lhs,
    a_rs=a_rs_lhs,
    inc=inc_lhs,
    rp=rp_lhs
)

In [ ]:
outputs_lhs = fitter_lhs.get_rw_outputs()

samples_lhs = outputs_lhs["samples"]
parameter_keys_lhs = outputs_lhs["parameter_keys"]
photometry_kwargs_lhs = outputs_lhs["photometry"]

In [ ]:
photometry_kwargs_lhs

# 3. Packaging results for the Data Challenge

Next up, we package the results obtained in the previous sections using the Rocky Worlds Data Challenge library `rocky_worlds_data_challenge`. Here we [basically follow the package tutorial](https://rocky-worlds-data-challenge.readthedocs.io/en/latest/tutorials/prepare-submission.html). Let's load the package:

In [ ]:
import rocky_worlds_data_challenge as rw

## 3.1 Packaging the posteriors

Given how we packaged our results in our fits before, this is very easy to do. The package expects the samples in a numpy array and the parameter keys in a list. We already have that, e.g., for LHS 1140 b:

In [ ]:
outputs_lhs["parameter_keys"]

In [ ]:
outputs_lhs["samples"].shape

So let's package that for both fits. To this we use the `rw.Posteriors` function which is a helper that does everything for us:

In [ ]:
posterior_GJ_3929_b = rw.Posterior(
    samples=outputs_gj["samples"],
    parameter_keys=outputs_gj["parameter_keys"],
)

posterior_LHS_1140_b = rw.Posterior(
    samples=outputs_lhs["samples"],
    parameter_keys=outputs_lhs["parameter_keys"],
)

In theory you can include any posterior you want here (the more the better!). Here we got lazy (such that our posterior output is not too large) and only gave the depths --- but ideally we would give the full posterior samples from all the parameters we fitted.

## 3.2 Packaging the photometry

Next up, we need to package the photometry. Here we need to package the entire setup from our model, the astrophysical model, the noise model, etc. The fitters we defined above already have all this information:

In [ ]:
plt.figure(figsize = (10,4))

plt.errorbar(photometry_kwargs_gj['time'], photometry_kwargs_gj['raw_flux'], yerr = photometry_kwargs_gj['raw_flux_err'], 
             fmt = '.', zorder = 1, alpha = 0.1)

tb, fb, _ = bin_by_points(photometry_kwargs_gj['time'], photometry_kwargs_gj['raw_flux'], bin_size=50)

plt.plot(tb, fb, 'o', mfc = 'white', mec = 'black', zorder = 2)

plt.plot(photometry_kwargs_gj['time'], photometry_kwargs_gj['astro_model'], 
             zorder = 3, label = 'Astrophysical model', color = 'orangered', lw = 3)

plt.plot(photometry_kwargs_gj['time'], photometry_kwargs_gj['noise_model'], 
             zorder = 4, label = 'Noise/systematics model', color = 'royalblue', lw = 3)

plt.plot(photometry_kwargs_gj['time'], photometry_kwargs_gj['full_model'], 
             zorder = 5, label = 'Full model', color = 'black')

plt.legend()

plt.xlabel('Time (BJD)', fontsize = 16)
plt.ylabel('Relative flux', fontsize = 16)

plt.xticks(fontsize = 14)
plt.yticks(fontsize = 14)

plt.ylim( 0.998,1.002 )
plt.xlim( np.min(photometry_kwargs_gj['time']), np.max(photometry_kwargs_gj['time']) )
plt.show()

Once again, we use some helper functions to package all of this up both for GJ 3929 b and LHS 1140 b:

In [ ]:
photometry_GJ_3929_b = rw.Photometry(

    time=photometry_kwargs_gj['time'],
    raw_flux=photometry_kwargs_gj['raw_flux'],
    raw_flux_err=photometry_kwargs_gj['raw_flux_err'],
    astro_model=photometry_kwargs_gj['astro_model'],
    noise_model=photometry_kwargs_gj['noise_model'],
    full_model=photometry_kwargs_gj['full_model'],
)

photometry_LHS_1140_b = rw.Photometry(

    time=photometry_kwargs_lhs['time'],
    raw_flux=photometry_kwargs_lhs['raw_flux'],
    raw_flux_err=photometry_kwargs_lhs['raw_flux_err'],
    astro_model=photometry_kwargs_lhs['astro_model'],
    noise_model=photometry_kwargs_lhs['noise_model'],
    full_model=photometry_kwargs_lhs['full_model'],
)

Note those are the required outputs, but if we had more parameters (e.g., photometry centroids, background flux, etc.) we could pass those too. In a real, full submission, those would be ideal to pass. Here we don't pass any of that because our photometry was obtained in the crudest way possible --- also to keep our output as small as possible. For more details on what you can pass, see [the package tutorial](https://rocky-worlds-data-challenge.readthedocs.io/en/latest/tutorials/prepare-submission.html).

## 3.3 Filling forms

The final input we need is to answer to a set of questions about what we did in our data reduction. This needs to be filled for both, GJ 3929 b reductions and LHS 1140 b ones. 

These forms can be filled by hand with a simple `json` file (if you are wondering "who is JSON?" [read this](https://stackoverflow.blog/2022/06/02/a-beginners-guide-to-json-the-data-format-for-the-internet/)) _or_ you can actually use the `rw` package itself to fill them in! You do this by running:

In [ ]:
rw.interactive_form()

## 3.4 Packaging all together

Finally, we package everything together:

In [ ]:
results = rw.Results(
    posterior_GJ_3929_b=posterior_GJ_3929_b,
    photometry_GJ_3929_b=photometry_GJ_3929_b,
    form_GJ_3929_b=rw.Form(path='form_GJ3929b.json'),
    posterior_LHS_1140_b=posterior_LHS_1140_b,
    photometry_LHS_1140_b=photometry_LHS_1140_b,
    form_LHS_1140_b=rw.Form(path='form_LHS1140b.json'),
)

In [ ]:
results.to_submission('submission.zip', overwrite=True)

Now you can submit this `submission.zip` to the Data Challenge!

<div class="alert alert-warning">
Please note our analyses above use a single eclipse observation for each target. The submitted results, thus, are unlikely to make it high in the leaderboard.
</div>

***

<table style="width:100%; border:0;">
<tr>
<td style="text-align:left; vertical-align:middle;">
<a href="#top">Top of Page</a>
</td>

<td style="text-align:right; vertical-align:middle;">
<img src="https://raw.githubusercontent.com/spacetelescope/roman_notebooks/7f391657bc2bf47a784400d65d2882703787416d/stsci_logo2.png"
     alt="Space Telescope Logo"
     width="80px"/>
</td>
</tr>
</table>